In [2]:
# ============================================================
# FRAGDENSTAAT:
# ALLE METADATEN BEREINIGEN UND DOWNLOAD-QUEUE ERSTELLEN
# ============================================================
#
# Dieser Code:
# - erkennt den Datenordner automatisch
# - lädt alle Metadaten aus CSV oder JSONL
# - bereinigt IDs, Titel, URLs, Dateigrößen und Seitenzahlen
# - entfernt ungültige Einträge und doppelte Download-Links
# - ergänzt Statusspalten für Download, Textextraktion und OCR
# - berechnet den geschätzten Speicherbedarf
# - speichert bereinigte Metadaten und Download-Warteschlange
#
# Es werden noch KEINE Dokumentdateien heruntergeladen.
# ============================================================

from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from IPython.display import display


# ============================================================
# 1. ARBEITS- UND DATENORDNER AUTOMATISCH ERKENNEN
# ============================================================

ARBEITSORDNER = Path.cwd().resolve()

if ARBEITSORDNER.name.lower() == "datenbank":
    PROJEKTORDNER = ARBEITSORDNER.parent
    DATENORDNER = ARBEITSORDNER

elif (ARBEITSORDNER / "Datenbank").is_dir():
    PROJEKTORDNER = ARBEITSORDNER
    DATENORDNER = ARBEITSORDNER / "Datenbank"

else:
    raise FileNotFoundError(
        "\nDer Datenordner konnte nicht automatisch gefunden werden.\n\n"
        f"Aktueller Arbeitsordner:\n{ARBEITSORDNER}\n\n"
        "Erwartet wurde entweder:\n"
        "- ein Arbeitsordner mit dem Namen 'Datenbank'\n"
        "- oder ein Unterordner namens 'Datenbank'"
    )


# ============================================================
# 2. DATEIPFADE FESTLEGEN
# ============================================================

QUELL_CSV = DATENORDNER / "fragdenstaat_alle_dokumente.csv"
QUELL_JSONL = DATENORDNER / "fragdenstaat_alle_dokumente.jsonl"

BEREINIGTE_CSV = (
    DATENORDNER
    / "fragdenstaat_alle_metadaten_bereinigt.csv"
)

BEREINIGTE_JSONL = (
    DATENORDNER
    / "fragdenstaat_alle_metadaten_bereinigt.jsonl"
)

DOWNLOAD_QUEUE_CSV = (
    DATENORDNER
    / "fragdenstaat_download_queue_all.csv"
)

DOWNLOAD_QUEUE_JSONL = (
    DATENORDNER
    / "fragdenstaat_download_queue_all.jsonl"
)


print("=" * 75)
print("FRAGDENSTAAT: ALLE METADATEN BEREINIGEN")
print("=" * 75)

print("\nArbeitsordner:")
print(ARBEITSORDNER)

print("\nProjektordner:")
print(PROJEKTORDNER)

print("\nVerwendeter Datenordner:")
print(DATENORDNER)

print("\nQuell-CSV vorhanden:")
print(QUELL_CSV.exists())

print("\nQuell-JSONL vorhanden:")
print(QUELL_JSONL.exists())


# ============================================================
# 3. VOLLSTÄNDIGE METADATEN LADEN
# ============================================================

if QUELL_CSV.exists():

    print("\nVollständige CSV wird geladen ...")

    df = pd.read_csv(
        QUELL_CSV,
        encoding="utf-8-sig",
        low_memory=False,
    )

    verwendete_datei = QUELL_CSV

elif QUELL_JSONL.exists():

    print("\nCSV nicht gefunden. JSONL wird geladen ...")

    df = pd.read_json(
        QUELL_JSONL,
        lines=True,
    )

    verwendete_datei = QUELL_JSONL

else:

    raise FileNotFoundError(
        "\nEs wurden keine vollständigen Metadaten gefunden.\n\n"
        "Erwartete Dateien:\n"
        f"- {QUELL_CSV}\n"
        f"- {QUELL_JSONL}"
    )


print("\nGeladene Datei:")
print(verwendete_datei)

print("\nUrsprüngliche DataFrame-Größe:")
print(f"{len(df):,} Zeilen und {df.shape[1]} Spalten")


# ============================================================
# 4. NOTWENDIGE SPALTEN PRÜFEN
# ============================================================

PFLICHTSPALTEN = [
    "id",
    "title",
    "file_url",
]

fehlende_spalten = [
    spalte
    for spalte in PFLICHTSPALTEN
    if spalte not in df.columns
]

if fehlende_spalten:
    raise KeyError(
        "\nFolgende notwendige Spalten fehlen:\n"
        + "\n".join(
            f"- {spalte}"
            for spalte in fehlende_spalten
        )
    )


# ============================================================
# 5. HILFSFUNKTIONEN
# ============================================================

def ist_gueltige_http_url(url: str) -> bool:
    """
    Prüft, ob ein Wert eine gültige HTTP- oder HTTPS-URL ist.
    """

    if not isinstance(url, str):
        return False

    url = url.strip()

    if not url:
        return False

    try:
        parsed = urlparse(url)

        return (
            parsed.scheme in {"http", "https"}
            and bool(parsed.netloc)
        )

    except Exception:
        return False


def to_boolean(series: pd.Series) -> pd.Series:
    """
    Wandelt unterschiedliche Wahr-/Falsch-Darstellungen
    zuverlässig in boolesche Werte um.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)

    normalized = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return (
        normalized
        .map(
            {
                "true": True,
                "1": True,
                "yes": True,
                "ja": True,
                "false": False,
                "0": False,
                "no": False,
                "nein": False,
                "": False,
                "nan": False,
                "none": False,
            }
        )
        .fillna(False)
        .astype(bool)
    )


def dateiendung_aus_url(url: str) -> str:
    """
    Liest die Dateiendung aus dem Pfad einer URL.
    """

    try:
        suffix = Path(
            urlparse(url).path
        ).suffix.lower()

        return suffix

    except Exception:
        return ""


def dokumenttyp_klassifizieren(
    extension: str,
) -> str:
    """
    Ordnet eine Dateiendung einer groben Dokumentklasse zu.
    """

    extension = str(extension).strip().lower()

    if extension == ".pdf":
        return "pdf"

    if extension in {
        ".doc",
        ".docx",
        ".odt",
        ".rtf",
        ".txt",
    }:
        return "document"

    if extension in {
        ".xls",
        ".xlsx",
        ".ods",
        ".csv",
    }:
        return "spreadsheet"

    if extension in {
        ".jpg",
        ".jpeg",
        ".png",
        ".tif",
        ".tiff",
        ".webp",
        ".bmp",
    }:
        return "image"

    if extension in {
        ".zip",
        ".rar",
        ".7z",
        ".tar",
        ".gz",
    }:
        return "archive"

    if extension == "":
        return "unknown"

    return "other"


def download_prioritaet(
    size_mb,
) -> str:
    """
    Ordnet Dokumente anhand der bekannten Dateigröße ein.
    """

    if pd.isna(size_mb):
        return "unknown_size"

    if size_mb <= 5:
        return "small"

    if size_mb <= 25:
        return "medium"

    if size_mb <= 100:
        return "large"

    return "very_large"


# ============================================================
# 6. ARBEITSKOPIE ERSTELLEN
# ============================================================

clean_df = df.copy()

urspruengliche_anzahl = len(clean_df)


# ============================================================
# 7. IDs BEREINIGEN
# ============================================================

clean_df["id"] = pd.to_numeric(
    clean_df["id"],
    errors="coerce",
)

anzahl_ungueltige_ids = (
    clean_df["id"].isna().sum()
)

clean_df = clean_df[
    clean_df["id"].notna()
].copy()

clean_df["id"] = (
    clean_df["id"]
    .astype("int64")
)


# ============================================================
# 8. TITEL BEREINIGEN
# ============================================================

clean_df["title"] = (
    clean_df["title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

clean_df.loc[
    clean_df["title"].eq(""),
    "title",
] = "Ohne Titel"


# ============================================================
# 9. DOWNLOAD-URLS BEREINIGEN UND PRÜFEN
# ============================================================

clean_df["file_url"] = (
    clean_df["file_url"]
    .fillna("")
    .astype(str)
    .str.strip()
)

clean_df["valid_file_url"] = (
    clean_df["file_url"]
    .apply(ist_gueltige_http_url)
)

anzahl_ungueltige_urls = int(
    (~clean_df["valid_file_url"]).sum()
)

clean_df = clean_df[
    clean_df["valid_file_url"]
].copy()


# ============================================================
# 10. BOOLEAN-SPALTEN NORMALISIEREN
# ============================================================

if "public" in clean_df.columns:
    clean_df["public"] = to_boolean(
        clean_df["public"]
    )
else:
    clean_df["public"] = True


if "pending" in clean_df.columns:
    clean_df["pending"] = to_boolean(
        clean_df["pending"]
    )
else:
    clean_df["pending"] = False


if "listed" in clean_df.columns:
    clean_df["listed"] = to_boolean(
        clean_df["listed"]
    )


if "allow_annotation" in clean_df.columns:
    clean_df["allow_annotation"] = to_boolean(
        clean_df["allow_annotation"]
    )


# Nur öffentliche und nicht ausstehende Dokumente behalten
anzahl_vor_public_filter = len(clean_df)

clean_df = clean_df[
    clean_df["public"]
    & ~clean_df["pending"]
].copy()

entfernt_durch_public_filter = (
    anzahl_vor_public_filter - len(clean_df)
)


# ============================================================
# 11. DATEIGRÖSSE BEREINIGEN
# ============================================================

if "file_size" in clean_df.columns:

    clean_df["file_size"] = pd.to_numeric(
        clean_df["file_size"],
        errors="coerce",
    )

else:

    clean_df["file_size"] = pd.NA


clean_df.loc[
    clean_df["file_size"] < 0,
    "file_size",
] = pd.NA


clean_df["file_size_mb"] = (
    clean_df["file_size"] / (1024 ** 2)
).round(2)

clean_df["file_size_gb"] = (
    clean_df["file_size"] / (1024 ** 3)
).round(4)


# ============================================================
# 12. SEITENZAHL BEREINIGEN
# ============================================================

if "num_pages" in clean_df.columns:

    clean_df["num_pages"] = pd.to_numeric(
        clean_df["num_pages"],
        errors="coerce",
    )

    clean_df.loc[
        clean_df["num_pages"] < 0,
        "num_pages",
    ] = pd.NA

else:

    clean_df["num_pages"] = pd.NA


# ============================================================
# 13. DATUMSSPALTEN BEREINIGEN
# ============================================================

if "last_modified_at" in clean_df.columns:

    clean_df["last_modified_at"] = pd.to_datetime(
        clean_df["last_modified_at"],
        errors="coerce",
        utc=True,
    )


if "published_at" in clean_df.columns:

    clean_df["published_at"] = pd.to_datetime(
        clean_df["published_at"],
        errors="coerce",
        utc=True,
    )


# ============================================================
# 14. DOPPELTE IDs ENTFERNEN
# ============================================================

anzahl_vor_id_duplikaten = len(clean_df)

clean_df = (
    clean_df
    .drop_duplicates(
        subset=["id"],
        keep="last",
    )
    .copy()
)

entfernte_id_duplikate = (
    anzahl_vor_id_duplikaten - len(clean_df)
)


# ============================================================
# 15. DOPPELTE DOWNLOAD-URLS ENTFERNEN
# ============================================================

anzahl_vor_url_duplikaten = len(clean_df)

clean_df = (
    clean_df
    .drop_duplicates(
        subset=["file_url"],
        keep="first",
    )
    .copy()
)

entfernte_url_duplikate = (
    anzahl_vor_url_duplikaten - len(clean_df)
)


# ============================================================
# 16. DATEIENDUNG UND DOKUMENTTYP ERMITTELN
# ============================================================

clean_df["file_extension"] = (
    clean_df["file_url"]
    .apply(dateiendung_aus_url)
)

clean_df["document_type"] = (
    clean_df["file_extension"]
    .apply(dokumenttyp_klassifizieren)
)


# ============================================================
# 17. DOWNLOAD-PRIORITÄT ERSTELLEN
# ============================================================

clean_df["download_priority"] = (
    clean_df["file_size_mb"]
    .apply(download_prioritaet)
)

prioritaets_reihenfolge = {
    "small": 1,
    "medium": 2,
    "large": 3,
    "very_large": 4,
    "unknown_size": 5,
}

clean_df["download_priority_order"] = (
    clean_df["download_priority"]
    .map(prioritaets_reihenfolge)
    .fillna(99)
    .astype(int)
)


# ============================================================
# 18. STATUSSPALTEN FÜR DIE SPÄTERE PIPELINE ANLEGEN
# ============================================================

clean_df["download_status"] = "pending"
clean_df["download_attempts"] = 0
clean_df["downloaded_at"] = ""
clean_df["local_file_path"] = ""
clean_df["http_status"] = pd.NA
clean_df["download_error"] = ""

clean_df["text_status"] = "not_started"
clean_df["text_extraction_method"] = ""
clean_df["text_file_path"] = ""
clean_df["text_error"] = ""

clean_df["ocr_required"] = pd.NA
clean_df["ocr_status"] = "not_checked"

clean_df["processing_status"] = "metadata_ready"


# ============================================================
# 19. DATEN SORTIEREN
# ============================================================

clean_df = (
    clean_df
    .sort_values(
        by=[
            "download_priority_order",
            "file_size",
            "id",
        ],
        ascending=[
            True,
            True,
            True,
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)


# ============================================================
# 20. BEREINIGTE METADATEN SPEICHERN
# ============================================================

print("\nBereinigte Gesamtdateien werden gespeichert ...")

clean_df.to_csv(
    BEREINIGTE_CSV,
    index=False,
    encoding="utf-8-sig",
)

clean_df.to_json(
    BEREINIGTE_JSONL,
    orient="records",
    lines=True,
    force_ascii=False,
    date_format="iso",
)


# ============================================================
# 21. DOWNLOAD-WARTESCHLANGE ERSTELLEN
# ============================================================

queue_spalten = [
    spalte
    for spalte in [
        "id",
        "uid",
        "title",
        "file_url",
        "site_url",
        "file_extension",
        "document_type",
        "num_pages",
        "file_size",
        "file_size_mb",
        "file_size_gb",
        "public",
        "pending",
        "last_modified_at",
        "download_priority",
        "download_priority_order",
        "download_status",
        "download_attempts",
        "downloaded_at",
        "local_file_path",
        "http_status",
        "download_error",
        "text_status",
        "text_extraction_method",
        "text_file_path",
        "text_error",
        "ocr_required",
        "ocr_status",
        "processing_status",
    ]
    if spalte in clean_df.columns
]

download_queue_df = (
    clean_df[queue_spalten]
    .copy()
)


download_queue_df.to_csv(
    DOWNLOAD_QUEUE_CSV,
    index=False,
    encoding="utf-8-sig",
)

download_queue_df.to_json(
    DOWNLOAD_QUEUE_JSONL,
    orient="records",
    lines=True,
    force_ascii=False,
    date_format="iso",
)


# ============================================================
# 22. SPEICHERBEDARF BERECHNEN
# ============================================================

bekannte_dateigroessen = (
    clean_df["file_size"]
    .dropna()
    .clip(lower=0)
)

gesamte_bytes = bekannte_dateigroessen.sum()

gesamte_gb = (
    gesamte_bytes / (1024 ** 3)
)

gesamte_tb = (
    gesamte_bytes / (1024 ** 4)
)

anzahl_ohne_groesse = int(
    clean_df["file_size"].isna().sum()
)


# ============================================================
# 23. ERGEBNISSE AUSGEBEN
# ============================================================

print("\n" + "=" * 75)
print("BEREINIGUNG ERFOLGREICH ABGESCHLOSSEN")
print("=" * 75)

print("\nUrsprüngliche Datensätze:")
print(f"{urspruengliche_anzahl:,}")

print("\nUngültige IDs:")
print(f"{anzahl_ungueltige_ids:,}")

print("\nUngültige oder fehlende Download-URLs:")
print(f"{anzahl_ungueltige_urls:,}")

print("\nDurch Public-/Pending-Filter entfernt:")
print(f"{entfernt_durch_public_filter:,}")

print("\nEntfernte doppelte IDs:")
print(f"{entfernte_id_duplikate:,}")

print("\nEntfernte doppelte Download-Links:")
print(f"{entfernte_url_duplikate:,}")

print("\nVerbleibende Dokumente:")
print(f"{len(clean_df):,}")

print("\nDokumente ohne bekannte Dateigröße:")
print(f"{anzahl_ohne_groesse:,}")

print("\nGeschätzter Speicherbedarf aller bekannten Dateien:")
print(f"{gesamte_gb:,.2f} GB")
print(f"{gesamte_tb:,.3f} TB")


# ============================================================
# 24. DATEITYPEN ANZEIGEN
# ============================================================

print("\n" + "=" * 75)
print("DATEITYPEN")
print("=" * 75)

dateitypen_df = (
    clean_df["document_type"]
    .value_counts(dropna=False)
    .rename_axis("document_type")
    .reset_index(name="anzahl")
)

display(dateitypen_df)


# ============================================================
# 25. DOWNLOAD-PRIORITÄTEN ANZEIGEN
# ============================================================

print("\n" + "=" * 75)
print("DOWNLOAD-PRIORITÄTEN")
print("=" * 75)

prioritaeten_df = (
    clean_df["download_priority"]
    .value_counts(dropna=False)
    .rename_axis("download_priority")
    .reset_index(name="anzahl")
)

display(prioritaeten_df)


# ============================================================
# 26. BEISPIELDATENSÄTZE ANZEIGEN
# ============================================================

anzeige_spalten = [
    spalte
    for spalte in [
        "id",
        "title",
        "document_type",
        "num_pages",
        "file_size_mb",
        "download_priority",
        "download_status",
        "file_url",
    ]
    if spalte in download_queue_df.columns
]

print("\n" + "=" * 75)
print("ERSTE 20 DOKUMENTE DER DOWNLOAD-WARTESCHLANGE")
print("=" * 75)

display(
    download_queue_df[
        anzeige_spalten
    ].head(20)
)


# ============================================================
# 27. SPEICHERORTE AUSGEBEN
# ============================================================

print("\n" + "=" * 75)
print("GESPEICHERTE DATEIEN")
print("=" * 75)

print("\nBereinigte Metadaten als CSV:")
print(BEREINIGTE_CSV)

print("\nBereinigte Metadaten als JSONL:")
print(BEREINIGTE_JSONL)

print("\nDownload-Warteschlange als CSV:")
print(DOWNLOAD_QUEUE_CSV)

print("\nDownload-Warteschlange als JSONL:")
print(DOWNLOAD_QUEUE_JSONL)

print("\nVerfügbare DataFrames:")
print("- clean_df")
print("- download_queue_df")

FRAGDENSTAAT: ALLE METADATEN BEREINIGEN

Arbeitsordner:
C:\Users\Admin\Desktop\OpenLens\Datenbank

Projektordner:
C:\Users\Admin\Desktop\OpenLens

Verwendeter Datenordner:
C:\Users\Admin\Desktop\OpenLens\Datenbank

Quell-CSV vorhanden:
True

Quell-JSONL vorhanden:
True

Vollständige CSV wird geladen ...

Geladene Datei:
C:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_alle_dokumente.csv

Ursprüngliche DataFrame-Größe:
258,184 Zeilen und 26 Spalten

Bereinigte Gesamtdateien werden gespeichert ...

BEREINIGUNG ERFOLGREICH ABGESCHLOSSEN

Ursprüngliche Datensätze:
258,184

Ungültige IDs:
0

Ungültige oder fehlende Download-URLs:
37

Durch Public-/Pending-Filter entfernt:
0

Entfernte doppelte IDs:
0

Entfernte doppelte Download-Links:
31

Verbleibende Dokumente:
258,116

Dokumente ohne bekannte Dateigröße:
0

Geschätzter Speicherbedarf aller bekannten Dateien:
189.79 GB
0.185 TB

DATEITYPEN


,document_type,anzahl
0,pdf,258089
1,other,27



DOWNLOAD-PRIORITÄTEN


,download_priority,anzahl
0,small,252110
1,medium,5244
2,large,643
3,very_large,119



ERSTE 20 DOKUMENTE DER DOWNLOAD-WARTESCHLANGE


,id,title,document_type,num_pages,file_size_mb,download_priority,download_status,file_url
0,167936,Bedienstete aus den alten Bundesländern in der...,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/b1/...
1,171514,Unterrichtsausfall aufgrund von Lehrermangel,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/be/...
2,84972,Aussagen von Minister Remmel zur Schulverpflegung,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/b8/...
3,85138,Haushaltsabschluss: Wofür hat Minister Remmel ...,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/ef/...
4,84216,Haushaltsabschluss: Wofür hat Minister Remmel ...,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/2d/...
5,84874,"Inobhutnahme und Beschulung unbegleiteter, min...",pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/4f/...
6,85976,Welche Probleme bestehen bei der Sicherung der...,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/5c/...
7,86393,Haushaltsabschluss: Wofür hat Minister Remmel ...,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/4a/...
8,87244,Haushaltsabschluss: Wofür hat Minister Remmel ...,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/48/...
9,88432,Haushaltsabschluss: Wofür hat Minister Remmel ...,pdf,1.0,0.0,small,pending,https://media.frag-den-staat.de/files/docs/b5/...



GESPEICHERTE DATEIEN

Bereinigte Metadaten als CSV:
C:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_alle_metadaten_bereinigt.csv

Bereinigte Metadaten als JSONL:
C:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_alle_metadaten_bereinigt.jsonl

Download-Warteschlange als CSV:
C:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_download_queue_all.csv

Download-Warteschlange als JSONL:
C:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_download_queue_all.jsonl

Verfügbare DataFrames:
- clean_df
- download_queue_df


In [3]:
# ============================================================
# BEREINIGUNGSERGEBNIS DETAILLIERT ANZEIGEN
# ============================================================
#
# Dieser Code:
# - fasst alle Bereinigungsschritte zusammen
# - zeigt absolute Zahlen und Prozentwerte
# - berechnet die insgesamt entfernten Datensätze
# - zeigt die verbleibenden Dokumente
# - erstellt eine übersichtliche Tabelle
# ============================================================

import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# 1. PRÜFEN, OB DIE BENÖTIGTEN VARIABLEN EXISTIEREN
# ------------------------------------------------------------

benoetigte_variablen = [
    "urspruengliche_anzahl",
    "anzahl_ungueltige_ids",
    "anzahl_ungueltige_urls",
    "entfernt_durch_public_filter",
    "entfernte_id_duplikate",
    "entfernte_url_duplikate",
    "clean_df",
]

fehlende_variablen = [
    variable
    for variable in benoetigte_variablen
    if variable not in globals()
]

if fehlende_variablen:
    raise NameError(
        "\nDie Bereinigung wurde noch nicht vollständig ausgeführt.\n"
        "Folgende Variablen fehlen:\n"
        + "\n".join(
            f"- {variable}"
            for variable in fehlende_variablen
        )
        + "\n\nFühre zuerst den vollständigen Bereinigungscode aus."
    )


# ------------------------------------------------------------
# 2. ANZAHL VERBLEIBENDER DATENSÄTZE
# ------------------------------------------------------------

verbleibende_datensaetze = len(clean_df)


# ------------------------------------------------------------
# 3. INSGESAMT ENTFERNTE DATENSÄTZE
# ------------------------------------------------------------

insgesamt_entfernt = (
    urspruengliche_anzahl
    - verbleibende_datensaetze
)


# ------------------------------------------------------------
# 4. PROZENTWERTE BERECHNEN
# ------------------------------------------------------------

def prozentwert(
    anzahl: int,
    gesamt: int,
) -> float:
    """
    Berechnet einen Prozentwert sicher.
    """

    if gesamt == 0:
        return 0.0

    return round(
        anzahl / gesamt * 100,
        4,
    )


prozent_insgesamt_entfernt = prozentwert(
    insgesamt_entfernt,
    urspruengliche_anzahl,
)

prozent_verbleibend = prozentwert(
    verbleibende_datensaetze,
    urspruengliche_anzahl,
)


# ------------------------------------------------------------
# 5. ÜBERSICHTSTABELLE ERSTELLEN
# ------------------------------------------------------------

bereinigungs_df = pd.DataFrame(
    {
        "Bereinigungsschritt": [
            "Ungültige IDs",
            "Ungültige oder fehlende Download-URLs",
            "Nicht öffentliche oder ausstehende Dokumente",
            "Doppelte IDs",
            "Doppelte Download-Links",
            "Insgesamt entfernt",
            "Verbleibende Dokumente",
        ],
        "Anzahl": [
            int(anzahl_ungueltige_ids),
            int(anzahl_ungueltige_urls),
            int(entfernt_durch_public_filter),
            int(entfernte_id_duplikate),
            int(entfernte_url_duplikate),
            int(insgesamt_entfernt),
            int(verbleibende_datensaetze),
        ],
    }
)

bereinigungs_df["Anteil an Ausgangsdaten (%)"] = (
    bereinigungs_df["Anzahl"]
    .apply(
        lambda wert: prozentwert(
            wert,
            urspruengliche_anzahl,
        )
    )
)


# ------------------------------------------------------------
# 6. GESAMTERGEBNIS AUSGEBEN
# ------------------------------------------------------------

print("=" * 75)
print("ERGEBNIS DER METADATEN-BEREINIGUNG")
print("=" * 75)

print("\nUrsprüngliche Datensätze:")
print(f"{urspruengliche_anzahl:,}")

print("\nInsgesamt entfernte Datensätze:")
print(f"{insgesamt_entfernt:,}")

print("\nVerbleibende Datensätze:")
print(f"{verbleibende_datensaetze:,}")

print("\nAnteil entfernt:")
print(f"{prozent_insgesamt_entfernt:.4f} %")

print("\nAnteil verbleibend:")
print(f"{prozent_verbleibend:.4f} %")


# ------------------------------------------------------------
# 7. DETAILTABELLE ANZEIGEN
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DETAILLIERTE BEREINIGUNGSSCHRITTE")
print("=" * 75)

display(
    bereinigungs_df
)


# ------------------------------------------------------------
# 8. KURZE ZUSAMMENFASSUNG
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("KURZE ZUSAMMENFASSUNG")
print("=" * 75)

print(
    f"\nVon ursprünglich {urspruengliche_anzahl:,} Datensätzen "
    f"wurden insgesamt {insgesamt_entfernt:,} Datensätze entfernt."
)

print(
    f"Es verbleiben {verbleibende_datensaetze:,} "
    f"bereinigte und nutzbare Dokumente."
)

print(
    f"Die Bereinigung hat "
    f"{prozent_insgesamt_entfernt:.4f} % "
    f"der Ausgangsdaten entfernt."
)

ERGEBNIS DER METADATEN-BEREINIGUNG

Ursprüngliche Datensätze:
258,184

Insgesamt entfernte Datensätze:
68

Verbleibende Datensätze:
258,116

Anteil entfernt:
0.0263 %

Anteil verbleibend:
99.9737 %

DETAILLIERTE BEREINIGUNGSSCHRITTE


,Bereinigungsschritt,Anzahl,Anteil an Ausgangsdaten (%)
0,Ungültige IDs,0,0.0000
1,Ungültige oder fehlende Download-URLs,37,0.0143
2,Nicht öffentliche oder ausstehende Dokumente,0,0.0000
3,Doppelte IDs,0,0.0000
4,Doppelte Download-Links,31,0.0120
5,Insgesamt entfernt,68,0.0263
6,Verbleibende Dokumente,258116,99.9737



KURZE ZUSAMMENFASSUNG

Von ursprünglich 258,184 Datensätzen wurden insgesamt 68 Datensätze entfernt.
Es verbleiben 258,116 bereinigte und nutzbare Dokumente.
Die Bereinigung hat 0.0263 % der Ausgangsdaten entfernt.
